# 07 - Noise processing with NoiseApp / YAWN (Python)

**File:** `notebooks/07_noise_processing_python.ipynb`

**What this does:** Turns raw hydrophone audio into calibrated soundscape
metrics with NoiseApp (YAWN): running the hybrid millidecade analysis over audio
on Google Cloud, reading the HDF5 files it writes, making SPD / LTSA /
third-octave plots, and exporting the numbers to CSV.

**How to run it:** Six worked examples from real analyses, not a notebook that
runs top to bottom. Find the section you want, copy the cell, and change the
paths and the Google Cloud bucket to point at your own data.

**Inputs:** audio on Google Cloud or on disk, plus a hydrophone calibration --
either a flat sensitivity in dB, or a CSV of frequency in Hz against
end-to-end calibration in dB.

**Outputs:** HDF5 files of band levels, figures, and CSV exports.

**Requires:** `h5py`, `scipy`, `seaborn`, `soundfile`, `numpy`, `matplotlib`
and -- for reading `gs://` buckets -- `google-cloud-storage`, all in
[`requirements.txt`](../requirements.txt). NoiseApp itself is in this repo, in
[`noiseprocessing/`](../noiseprocessing/); the setup cell below makes it
importable.

**See also:** [`08_pypam_validation_python.ipynb`](08_pypam_validation_python.ipynb),
which checks these outputs against PyPAM.

---

## Contents

| # | Section |
|---|---|
| 1 | Run the analysis on one deployment |
| 2 | Open an HDF5 and export to CSV |
| 3 | Batch: every Glider Rodeo deployment |
| 4 | What is in my HDF5? Plot options and CSV |
| 5 | WHICEAS 2020 DASBR survey |
| 6 | WHICEAS 2026 DASBR survey |

Sections 1 and 2 are the shortest way in -- build a `NoiseApp`, call
`run_analysis()`, then open the resulting HDF5 and plot it. Section 3 does that
over many deployments at once and 5 and 6 do the same for drifting recorders.
Section 4 is the guided tour of what you can do with an HDF5 once you have one.

## Setup

The noise analysis package lives in [`noiseprocessing/`](../noiseprocessing/) at
the top of this repo, so there is nothing to clone or install for it. Run this
cell first -- it puts that folder on `sys.path`, which is what lets the
`from noiseProcessGoogleCloud import ...` lines below work.

In [ ]:
# Make the vendored NoiseApp (YAWN) package importable.
# It lives in noiseprocessing/ at the top of this repo -- see that folder's
# README for where it came from. This works wherever you run the notebook from.
import sys
from pathlib import Path

for _root in [Path.cwd(), *Path.cwd().parents]:
    _pkg = _root / "noiseprocessing"
    if _pkg.is_dir():
        if str(_pkg) not in sys.path:
            sys.path.insert(0, str(_pkg))
        break
else:
    raise RuntimeError(
        "Could not find noiseprocessing/ -- open this notebook from inside the repo."
    )

print("NoiseApp will be imported from:", _pkg)

---

## 1. Run the analysis on one deployment

The smallest complete example: point `NoiseApp` at a folder of audio on Google
Cloud, give it a calibration and somewhere to write, and run it. Then open the
HDF5 it wrote and plot a day, plot many days, and poke at the raw arrays.

### 1.1 Set up and run

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Dec 10 18:26:22 2025

@author: pam_user
"""
from noiseProcessGoogleCloud import NoiseApp


# Define the location of the data on google cloud
gsCloudLoc = "gs://swfsc-1/2024_CalCurCEAS/glider/audio_flac/sg680_CalCurCEAS_Sep2024"


# Define where you want to store the data
out_dir = r"C:\Users\pam_user\Documents\HybridMilliDaily"

# Loccation of the calibration csv, first column should be frequency in Hz 
# Second column should be end-to-end calibration in dB
calib_csv = 'C:\\Users\\pam_user\\Downloads\\sg680_CalCurCEAS_Sep2024_sensitivity_2025-07-29.csv'


# Declare the noise app object and give it a project name and 
# a deployment name. Note that you can store multiple 
# deployments within a project
app = NoiseApp(
    Si=calib_csv,
    soundFilePath=gsCloudLoc,
    ProjName='sg680_CalCurCEAS_Apr2022',
    DepName='SG680',
    DatabaseLoc=out_dir,
    rmDC=True, # Remove the DC offset from each audio file
    Si_units='V/µPa'
)

# Go do the thing!
app.run_analysis()

### 1.2 Plot one day

In [ ]:
#%% Plot example day

import h5py
import glob
from pathlib import Path
from noiseProcessGoogleCloud import plot_milidecade_statistics, plot_ltsa

#Example for plotting (uncomment and point to an HDF5 from out_dir)
h5_path = r"X:\\\\Kaitlin_Palmer\\\\CalCursea_680_Noise\\\\sg680_CalCurCEAS_Sep2024_20241001.h5"


# With the included plotting function, make a plot
with h5py.File(h5_path, 'r') as hdf_file:
    Project = hdf_file['CalCurCEAS_2024']
    plot_milidecade_statistics(Project) # This takes a while
    fig = plot_ltsa(Project, 
                    averaging_period='1hr',
                    title="LTSA – Day 1 1hr Resolution",
                    freq_scaled=True,   # real frequency on y
                    log_freq=True)
    
    
    
    

### 1.3 Plot 53 days

In [ ]:
#%% Plot example days

# Plot multiple
data_dir = r"X:\\\\Kaitlin_Palmer\\\\CalCursea_680_Noise\\\\"
paths = sorted(Path(data_dir).glob("*.h5"))[3:60]  # adjust pattern
groups = []

for p in paths:
    h5 = h5py.File(p, "r")
    # adjust group name as appropriate for your files
    groups.append(h5["CalCurCEAS_2024"])
    

#plot_milidecade_statistics(groups, pBands=[5, 25, 50, 75, 95])
fig = plot_ltsa(groups, 
                averaging_period='1d', # one day
                title="LTSA – 53 days 1 day resolution",
                freq_scaled=True,   # real frequency on y
                log_freq=True)

### 1.4 Look at the arrays directly

In [ ]:
#%% Data exploration


# Explore the hdf5 file a bit
hdf_file = h5py.File(h5_path, 'r')

# This should be the project name
projectName = list(hdf_file.keys())

# Use this to see the deployments within the project
hdf_file[projectName[0]].keys()

# This shows you the various metrics  including datetime stamp, broadband
# decadd, third octave, and hybridmilidecade band levels 


# Lets look at the first  ten decade levels and their frequencies
hdf_file[projectName[0]]['decadeLevels'][0:9] # Values
hdf_file[projectName[0]]['decadeFreqHz'][0:9] # Lower frequency range

# Lets look at the first  ten decade levels and their frequencies
hdf_file[projectName[0]]['hybridMiliDecLevels'][0:10] # Values
hdf_file[projectName[0]]['hybridDecFreqHz'][0:10] # Lower frequency range

---

## 2. Open an HDF5 and export to CSV

No processing here -- this one starts from an HDF5 that already exists, walks
its groups, and writes the broadband metric out as CSV for each project.

### 2.1 Export broadband levels per project

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Dec 10 18:26:22 2025

@author: pam_user
"""


#%% Plot example day

import h5py
import glob
from pathlib import Path
from noiseProcessGoogleCloud import plot_milidecade_statistics, plot_ltsa, plot_third_octave_bands, export_metric_csv

#Example for plotting (uncomment and point to an HDF5 from out_dir)
h5_path = r"C:/Users\\pam_user\\Downloads\\JavaInlet.h5"


# Explore the hdf5 file a bit
hdf_file = h5py.File(h5_path, 'r')

# This should be the project name
projectNames = list(hdf_file.keys())

# Use this to see the deployments within the project
hdf_file[projectNames[0]].keys()



# With the included plotting function, make a plot
for proj in projectNames:
    with h5py.File(h5_path, 'r') as hdf_file:
        Project = hdf_file[proj]
        #plot_milidecade_statistics(Project, title=proj) # This takes a while
        #plot_third_octave_bands(hdf_file[proj])
        export_metric_csv(h5_path, metric = 'broadband', group_name = proj, output_csv= proj+'broadband.csv' )
    
    
    

### 2.2 Data exploration

In [ ]:
#%% Data exploration


# Explore the hdf5 file a bit
hdf_file = h5py.File(h5_path, 'r')

# This should be the project name
projectName = list(hdf_file.keys())

# Use this to see the deployments within the project
hdf_file[projectName[0]].keys()

# This shows you the various metrics  including datetime stamp, broadband
# decadd, third octave, and hybridmilidecade band levels 


# Lets look at the first  ten decade levels and their frequencies
hdf_file[projectName[0]]['decadeLevels'][0:9] # Values
hdf_file[projectName[0]]['decadeFreqHz'][0:9] # Lower frequency range

# Lets look at the first  ten decade levels and their frequencies
hdf_file[projectName[0]]['hybridMiliDecLevels'][0:10] # Values
hdf_file[projectName[0]]['hybridDecFreqHz'][0:10] # Lower frequency range

---

## 3. Batch: every Glider Rodeo deployment

The same run as section 1, but over a list of deployments. Each entry keeps its
bucket path, channel and calibration together so they cannot drift apart. Note
`local_staging_dir` -- HDF5 writes go to a local scratch drive and sync to the
shared drive every 25 files, which is much faster than writing over the network.

The second cell then makes an SPD and an LTSA for each deployment.

### 3.1 Process all deployments

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Dec 10 18:26:22 2025

@author: pam_user
"""
from noiseProcessGoogleCloud import NoiseApp, print_h5_tree
import os
import numpy as np



# One entry per glider deployment, keeping all related settings together
# so they can't accidentally get out of sync with each other.

calib_csv_Whispr = 'C:\\Users\\pam_user\\Documents\\GitHub\\SPACIOUS-Propagation-Modes\\ExampleData\\sg680_CalCurCEAS_Sep2024_sensitivity_2025-07-29.csv'
SeaExplorer_calib = "C:\\Users\\pam_user\\Downloads\SEA117-M026_20260128_hpSensitivity_ch1_ESTIMATE.csv"


deployments = [
    {
        "mission_id": "sg607_20260128",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/sg607_20260128_WHICEAS/Recordings_CENSOR/flac",
        "channel": 1, # run channel 1
        "hyd_sensitivity": calib_csv_Whispr,  # HTI calibration
    },
    {
        "mission_id": "sg274_20260128",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/sg274_20260128_WHICEAS/Recordings_CENSOR/flac",
        "channel": 1,
        "hyd_sensitivity": calib_csv_Whispr,  # HTI calibration
    },
    
    {
        "mission_id": "risso-20260128",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/risso-20260128/Recordings_CENSOR/wav_2kHz",
        "channel": 1,
        "hyd_sensitivity": -203,
     },
    
    {
        "mission_id": "stenella-20260128",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/stenella-20260128/Recordings_CENSOR/flac",
        "channel": 1,
        "hyd_sensitivity": -165,  # HTI calibration
    },    
    
    {
        "mission_id": "capex987_20260128",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/capex987_20260128/Recordings_CENSOR/wav_512kHz/OBS-1195.17.512000.M36-V35-100",
        "channel": 1,
        "hyd_sensitivity": -165.11,  # HTI calibration
    }, 
    
    {
        "mission_id": "belladonna_20260128",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/belladonna_20260128/Recordings_CENSOR/flac/200kHz",
        "channel": 1,
        "hyd_sensitivity":  -176,
        },
    
    {
        "mission_id": "SEA117-M026_20260128_30sec",
        "gs_path": "gs://nmfs-collaborative/2026_GliderRodeo/SEA117-M026_20260128/Recordings_CENSOR/wav_30s",
        "channel": 1,
        "hyd_sensitivity": SeaExplorer_calib  
        },
    
    
]

# Define where you want to store the data
out_dir_base = r"X:\Kaitlin_Palmer\GliderRodeo"

# Local scratch drive for staging HDF5 writes before syncing to the shared X: drive
local_staging_base = r"C:\GliderRodeoScratch"

for deployment in deployments:

    print(deployment["gs_path"])

    out_dir = os.path.join(out_dir_base, deployment["mission_id"])
    staging_dir = os.path.join(local_staging_base, deployment["mission_id"])

    

    # Declare the noise app object and give it a project name and
    # a deployment name. Note that you can store multiple
    # deployments within a project
    app = NoiseApp(
        soundFilePath=deployment["gs_path"],
        ProjName=deployment["mission_id"],
        DepName='GliderRodeo',
        channel=deployment["channel"],
        Si=deployment["hyd_sensitivity"],
        DatabaseLoc=out_dir,
        split_hdf5_by_day=False,
        rmDC=True, # Remove the DC offset from each audio file
        Si_units='V/µPa',
        existing_deployment_mode='skip',
        local_staging_dir=staging_dir,
        sync_every_n_files=25)
    
    # Confirm there's audio to process before kicking off the analysis
    audio_files = app._list_audio_inputs()
    print(f"Found {len(audio_files)} audio file(s) in {deployment['gs_path']}")
    if not audio_files:
        print(f"Skipping {deployment['mission_id']}: no audio files found.")
        continue
    
    

    # Go do the thing!
    app.run_analysis()
    
    

### 3.2 Create PSD plots

In [ ]:
#%% Create PSD plots

import h5py
import glob
from pathlib import Path
from noiseProcessGoogleCloud import plot_milidecade_statistics, plot_ltsa, plot_third_octave_bands
import os
import matplotlib.pyplot as plt

#Example for plotting (uncomment and point to an HDF5 from out_dir)





# Where to store the figures
figDir = r"X:\\Kaitlin_Palmer\\GliderRodeo\\TestFigures\\"


# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\SEA117-M026_20260128_30sec\\SEA117-M026_20260128_30sec.h5"
# Oh no! I forget the structure of the HDF5!
print_h5_tree(h5_path)




hdf_file = h5py.File(h5_path, 'r')
Glider_id = "M026_20260128_30sec"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")

print_h5_tree(h5_path)

fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)



save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)


plt.close(fig)



# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\belladonna_20260128\\belladonna_20260128.h5"
hdf_file = h5py.File(h5_path, 'r')
Glider_id = "belladonna_20260128"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")
fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)
plt.close(fig)




# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\capex987_20260128\\capex987_20260128.h5"
hdf_file = h5py.File(h5_path, 'r')
Glider_id = "capex987_20260128"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")
fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)
plt.close(fig)




# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\risso-20260128\\risso-20260128.h5"
hdf_file = h5py.File(h5_path, 'r')
Glider_id = "risso-20260128"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")
fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)
plt.close(fig)



# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\sg274_20260128\\sg274_20260128.h5"
hdf_file = h5py.File(h5_path, 'r')
Glider_id = "sg274_20260128"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")
fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)
plt.close(fig)




# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\sg607_20260128\\sg607_20260128.h5"
hdf_file = h5py.File(h5_path, 'r')
Glider_id = "sg607_20260128"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")
fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)
plt.close(fig)




# Explore the hdf5 file a bit
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\stenella-20260128\\stenella-20260128.h5"
hdf_file = h5py.File(h5_path, 'r')
Glider_id = "stenella-20260128"
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.png")
fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, 
                                 save_path=save_file)  # This takes a while
plt.close(fig)
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")
fig = plot_ltsa(hdf_file['GliderRodeo'], title=Glider_id, save_path=save_LTSA,
                averaging_period='5min',
                freq_scaled=True,   # real frequency on y
                log_freq=True)
plt.close(fig)

---

## 4. What is in my HDF5? Plot options and CSV

The walkthrough. How to recover the structure of an HDF5 you wrote weeks ago
(`summarize_hdf5_file`, `list_hdf5_deployments`, `print_h5_tree`), the full set
of plotting options with their arguments explained, and how to get the numbers
out as CSV.

This is the section to read first if you are not sure what you are looking at.

### 4.1 Remind yourself what is in the file, then plot it

In [ ]:
import h5py
import glob
from pathlib import Path
from noiseProcessGoogleCloud import (print_h5_tree, plot_milidecade_statistics, 
                                     plot_ltsa, plot_third_octave_bands,
                                     list_hdf5_deployments, summarize_hdf5_file,
                                     plot_third_octave_bands,
                                     export_metric_csv)
import os
import matplotlib.pyplot as plt



# Imagine you've been running the noise analysis for a few days and thought
# quite carefully about how you set it up but now you can't recall. The following
# functions are designed to display some of the basics of the noise file. 

# Where you stored your favoirite glider file
h5_path = r"X:\Kaitlin_Palmer\GliderRodeo\sg607_20260128\\sg607_20260128.h5"
summarize_hdf5_file(h5_path) # The basics of the data and dataset names (e.g. OH CRAP I FORGET)
figDir = r"X:\\Kaitlin_Palmer\\GliderRodeo\\TestFigures\\"


# I only want info on one dataset in the HDF5
summarize_hdf5_file(h5_path, group_name='GliderRodeo')

# I only need the deployment names (because I forget just those)
list_hdf5_deployments(h5_path)


# Plots are available for one dataset (i.e. deployment at a time) so you
# should know the deployment id or use list_hdf5_deployments to recover them
# if you want to string multiple together


hdf_file = h5py.File(h5_path, 'r')
Glider_id = "sg607_20260128"

# Output path and name comprised of several parts, for simplicity you could
# use just one string
save_file = os.path.join(figDir, f"{Glider_id}_milidecade_SPD.pfd")

fig = plot_milidecade_statistics(hdf_file['GliderRodeo'], 
                                 title=Glider_id, # If you want a custom title
                                 save_path=save_file,   
                                 pBands=[5, 25, 50, 75, 95], # Probability Bands to show
                                 dpi= 150)  # Resolution (for publicaiton figures)

# This takes a while
plt.close(fig)

# Create an LTSA
save_LTSA = os.path.join(figDir, f"{Glider_id}_5min_ltsa.png")

fig = plot_ltsa(hdf_file['GliderRodeo'], 
                title=Glider_id, 
                save_path=save_LTSA,
                averaging_period='8min',  #Pandas offset alias for time-averaging (e.g., '5min', '1min', '1H',12d).
                freq_scaled=True,   # real frequency on y
                log_freq=False,
                dpi=150)
plt.close(fig)


# You can also just tell it to make a plot
plot_third_octave_bands(hdf_file['GliderRodeo'])

### 4.2 Heck with Python, I want these data in a CSV

In [ ]:
#%% Heck with python, I want these data in a CSV

projectNames = list_hdf5_deployments(h5_path)

# With the included plotting function, make a plot
for proj in projectNames:
    with h5py.File(h5_path, 'r') as hdf_file:
        Project = hdf_file[proj]
        #plot_milidecade_statistics(Project, title=proj) # This takes a while
        #plot_third_octave_bands(hdf_file[proj])
        export_metric_csv(h5_path, 
                          metric = 'broadband', 
                          group_name = proj, 
                          utput_csv= proj+'broadband.csv' )

---

## 5. WHICEAS 2020 DASBR survey

Thirteen drifting recorders, each with its own HTI hydrophone sensitivity and
serial number. The deployment id is built from the bucket path (`DS1` becomes
`DS01_ch01_hti_856081`) so the HDF5 groups stay sortable and self-describing.
All of them go into a single project HDF5.

### 5.1 Process the 2020 drifters

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Dec 10 18:26:22 2025

@author: pam_user
"""
from noiseProcessGoogleCloud import NoiseApp, print_h5_tree
import os
import numpy as np

# Define the location of the data on google cloud
gsCloudLoc = [
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS1/ST-1/1208795167",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS2/ST-2/1208766495",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS3/ST-15/470290496",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS4/ST-3/1208520735",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS5/ST-4/1208504351",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS6/ST-5/1208754207",# bad data
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS7/ST-7/1208496160",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS8/ST-8/1208774686",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS9/ST-9/1208487968",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS11/ST-11/470081600",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS12/ST-12/671125543",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS13/ST-13/470077504",
    "gs://pifsc-1/drifting_recorder/2020_WHICEAS_2001/DS14/ST-2/1208766495"]




hydSen__Ch1 =[-164.9,
-155.0,
-155.2,
-155.0,
-155.0,
-155.0,
-155.4,
-155.0,
-164.9,
-164.9,
-155.0,
-155.4,
-155.0,
-155.4]

ht1SericalNumber_Ch1 = [856081,
856091,
856116,
856091,
856087,
856091,
856086,
856087,
856081,
856081,
856087,
856086,
856087,
856086]



# Define where you want to store the data
out_dir = r"X:\Kaitlin_Palmer\WHICEASE2020_soundscape"

# 9,10,13,
for ii in range(0,len(gsCloudLoc)):
    print(gsCloudLoc[ii]) 
    path = gsCloudLoc[ii]
    ds_part = [p for p in path.split('/') if p.startswith('DS')][0]
    ds_num = int(ds_part[2:])
    ds_str = f"DS{ds_num:02d}"
    
    depId = ds_str+"_ch01_hti_"+str(ht1SericalNumber_Ch1[ii])
    print(depId) 
    
    # Declare the noise app object and give it a project name and 
    # a deployment name. Note that you can store multiple 
    # deployments within a project
    app = NoiseApp(
        soundFilePath=gsCloudLoc[ii],
        ProjName='WHICEAS2020_ch01',
        DepName=depId,
        channel = 0,
        Si = hydSen__Ch1[ii], # HTI ccalibration
        DatabaseLoc=out_dir,
        split_hdf5_by_day = False,
        rmDC=True, # Remove the DC offset from each audio file
        Si_units='V/µPa',
        existing_deployment_mode='overwrite')

    # Go do the thing!
    app.run_analysis()

### 5.2 Create PSD plots

In [ ]:
#%% Create PSD plots

import h5py
import glob
from pathlib import Path
from noiseProcessGoogleCloud import plot_milidecade_statistics, plot_ltsa, plot_third_octave_bands
import os
import matplotlib.pyplot as plt

#Example for plotting (uncomment and point to an HDF5 from out_dir)
h5_path = r"X:\Kaitlin_Palmer\WHICEASE2020_soundscape\WHICEAS2020_ch01.h5"

# Where to store the figures
figDir = r"X:\Kaitlin_Palmer\WHICEASE2020_soundscape\figures\\"

# Explore the hdf5 file a bit
hdf_file = h5py.File(h5_path, 'r')

# This should be the project name
deploymentNames = list(hdf_file.keys())


# With the included plotting function, make a plot

# With the included plotting function, make a plot

with h5py.File(h5_path, 'r') as hdf_file:
    for ii in range(len(deploymentNames)):
        dep_name = deploymentNames[ii]
        Project = hdf_file[dep_name]
        
        DASBR_id = dep_name.split('_',1)[0]
        
        save_file = os.path.join(figDir, f"{DASBR_id}_milidecade_SPD.png")
        fig = plot_milidecade_statistics(Project, 
                                         title=DASBR_id, 
                                         save_path=save_file)  # This takes a while
        plt.close(fig)
        save_tob = os.path.join(figDir, f"{DASBR_id}_third_octave.png")
        fig = plot_third_octave_bands(Project, 
                                      title=DASBR_id, 
                                      save_path=save_tob)
        plt.close(fig)
        save_LTSA = os.path.join(figDir, f"{DASBR_id}_5min_ltsa.png")
        fig = plot_ltsa(Project, title=DASBR_id, save_path=save_LTSA,
                        averaging_period='5min',
                        freq_scaled=True,   # real frequency on y
                        log_freq=False)
        plt.close(fig)

---

## 6. WHICEAS 2026 DASBR survey

The same pattern for the 2026 survey -- more drifters, and note
`existing_deployment_mode='skip'` rather than `'overwrite'`, so re-running does
not redo work already done.

### 6.1 Process the 2026 drifters

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Dec 10 18:26:22 2025

@author: pam_user
"""
from noiseProcessGoogleCloud import NoiseApp, print_h5_tree
import os
import numpy as np

# Define the location of the data on google cloud
gsCloudLoc = [
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS1/ST-4/1208504351",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS2/ST-9/1208487968",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS3/ST-11/470081600",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS5/ST-1/1208795167",
    #"gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS6-StillOut",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS7/ST-2/1208766495", 
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS8/ST-7/1208496160", 
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS8/ST-7/1208496160", 
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS9/ST-5/1208754207",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS10_noData",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS11/ST-1/1208795167",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS12/ST-13/470077504",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS13/ST-2/1208766495",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS14/ST-4/1208504351",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS15/ST-17/5982",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS16/ST-5/1208754207",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS17/ST-3/1208520735",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS18/ST-7/1208496160",
    "gs://pifsc-1/drifting_recorder/2026_WHICEAS_2601/DS19/ST-9/1208487968"]




hydSen__Ch1 =[-155.4,
-155.3,
-155.4,
-155.0,
#-155.6,
-155.3,
-155.4,
-155.3,
-155.0,
-155.0,
-155.4,
-155.3,
-155.3,
-155.6,
-155.4,
-155.4,
-154.6,
-155.3,
-155.7]

ht1SericalNumber_Ch1 = [856109,
856114,
856086,
856087,
#856113,
856114,
856109,
856115,
856085,
856085,
856109,
856115,
856115,
856113,
856109,
856086,
856088,
856131,
856111]



# Define where you want to store the data
out_dir = r"X:\Kaitlin_Palmer\WHICEASE2026_soundscape"

# 9,10,13,
for ii in range(0,len(gsCloudLoc)):
    print(gsCloudLoc[ii]) 
    path = gsCloudLoc[ii]
    ds_part = [p for p in path.split('/') if p.startswith('DS')][0]
    ds_num = int(ds_part[2:])
    ds_str = f"DS{ds_num:02d}"
    
    depId = ds_str+"_ch01_hti_"+str(ht1SericalNumber_Ch1[ii])
    print(depId) 
    
    # Declare the noise app object and give it a project name and 
    # a deployment name. Note that you can store multiple 
    # deployments within a project
    app = NoiseApp(
        soundFilePath=gsCloudLoc[ii],
        ProjName='WHICEAS2026_ch01',
        DepName=depId,
        channel = 0,
        Si = hydSen__Ch1[ii], # HTI ccalibration
        DatabaseLoc=out_dir,
        split_hdf5_by_day = False,
        rmDC=True, # Remove the DC offset from each audio file
        Si_units='V/µPa',
        existing_deployment_mode='skip')

    # Go do the thing!
    app.run_analysis()

### 6.2 Create PSD plots

In [ ]:
#%% Create PSD plots

import h5py
import glob
from pathlib import Path
from noiseProcessGoogleCloud import plot_milidecade_statistics, plot_ltsa, plot_third_octave_bands
import os
import matplotlib.pyplot as plt

#Example for plotting (uncomment and point to an HDF5 from out_dir)
h5_path = r"X:\Kaitlin_Palmer\WHICEASE2026_soundscape\WHICEAS2026_ch01.h5"

# Where to store the figures
figDir = r"X:\Kaitlin_Palmer\WHICEASE2026_soundscape\figures\\"

# Explore the hdf5 file a bit
hdf_file = h5py.File(h5_path, 'r')

# This should be the project name
deploymentNames = list(hdf_file.keys())


# With the included plotting function, make a plot
with h5py.File(h5_path, 'r') as hdf_file:
    for ii in range(len(deploymentNames)):
        dep_name = deploymentNames[ii]
        Project = hdf_file[dep_name]
        
        DASBR_id = dep_name.split('_',1)[0]
        
        save_file = os.path.join(figDir, f"{DASBR_id}_milidecade_SPD.png")
        fig = plot_milidecade_statistics(Project, 
                                         title=DASBR_id, 
                                         save_path=save_file)  # This takes a while
        plt.close(fig)
        save_tob = os.path.join(figDir, f"{DASBR_id}_third_octave.png")
        fig = plot_third_octave_bands(Project, 
                                      title=DASBR_id, 
                                      save_path=save_tob)
        plt.close(fig)
        save_LTSA = os.path.join(figDir, f"{DASBR_id}_5min_ltsa.png")
        fig = plot_ltsa(Project, 
                        title=DASBR_id, 
                        save_path=save_LTSA,
                        averaging_period='5min',
                        freq_scaled=True,   # real frequency on y
                        log_freq=False)
        plt.close(fig)